In [2]:
# 加载环境变量
from dotenv import load_dotenv
import os

load_dotenv()
qwen_api_key = os.getenv("aliyun")

# 1.没有记忆时

In [3]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 创建qwen模型
qwen_model = init_chat_model(
    model='qwen-max',
    model_provider='openai',
    base_url='https://maas.qianwenaiapi.com/compatible-mode/v1',
    api_key=qwen_api_key,
)

# 创建deepseek模型
# agent = create_agent(model = "deepseek-chat")


In [4]:
from langchain.messages import HumanMessage

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫虎哥，我最喜欢猫猫。")]}
)

print(response)

NameError: name 'agent' is not defined

In [5]:
# 第二次调用，询问我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="我最喜欢什么动物?")]}
)

print(response)

NameError: name 'agent' is not defined

# 2.添加记忆

添加会话记忆（短期记忆）分为三步：
- 导入并初始化Checkpointer
- 创建Agent，指定Checkpointer
- 调用Agent，指定thread_id


In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver：状态仅存在于 Python 进程的内存中，服务一旦重启或崩溃，所有会话记忆会立即全部丢失

# 创建agent
agent = create_agent(
    model=qwen_model,
    tools=[],
    checkpointer=InMemorySaver(),
)

In [7]:
from langchain.messages import HumanMessage

# 设定thread_id,作为会话标识，把配置项初始化为一个变量
config = {
    "configurable": {
        "thread_id": "user_1",
    }
}

# 调用模型
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫虎哥，我最喜欢猫猫。")]},
    config=config,          # 调用时多传一个配置项，这样所有基于user_1的调用都会被记录，作为同一组的会话快照
)
print(response)

{'messages': [HumanMessage(content='你好，我叫虎哥，我最喜欢猫猫。', additional_kwargs={}, response_metadata={}, id='3c0aa96f-f3a4-4b47-9f56-97992f49df41'), AIMessage(content='你好虎哥！很高兴认识你。喜欢猫猫是一件很温馨的事情呢，猫咪们通常都很可爱、聪明，能给人们带来许多欢乐。你有没有特别喜欢的猫咪品种或者关于养猫的经验想要分享的？如果需要关于如何更好地照顾猫咪的一些建议，我也很乐意帮助提供信息哦！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 20, 'total_tokens': 88, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'qwen-max', 'system_fingerprint': None, 'id': 'chatcmpl-e726fe5a-3b0b-9f62-a740-6e058c8224d1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a18893-3f49-76d1-ae51-2fb6214451f2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 68, 'total_tokens': 88, 'input_token_details': {'cache_read': 0}, 'output_token_deta

In [8]:
response = agent.invoke({"messages": [HumanMessage(content="我最喜欢什么动物?")]}, config=config)
print(response)

{'messages': [HumanMessage(content='你好，我叫虎哥，我最喜欢猫猫。', additional_kwargs={}, response_metadata={}, id='3c0aa96f-f3a4-4b47-9f56-97992f49df41'), AIMessage(content='你好虎哥！很高兴认识你。喜欢猫猫是一件很温馨的事情呢，猫咪们通常都很可爱、聪明，能给人们带来许多欢乐。你有没有特别喜欢的猫咪品种或者关于养猫的经验想要分享的？如果需要关于如何更好地照顾猫咪的一些建议，我也很乐意帮助提供信息哦！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 20, 'total_tokens': 88, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'qwen-max', 'system_fingerprint': None, 'id': 'chatcmpl-e726fe5a-3b0b-9f62-a740-6e058c8224d1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a18893-3f49-76d1-ae51-2fb6214451f2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 68, 'total_tokens': 88, 'input_token_details': {'cache_read': 0}, 'output_token_deta

# 3.Memory持久化存储

这里我们选择使用Sqlite作为存储方案，首先需要按照langgraph-checkpoint-sqlite依赖：
```
uv add langgraph-checkpoint-sqlite
```
接着，按照以下步骤使用： 一共四步
- 导入依赖
- 初始化checkpointer
- 自动建表
- 创建Agent，指定checkpointer


In [14]:
import sqlite3              # sqlite3 是 Python 内置的轻量级数据库，不需要额外安装服务，数据存在一个 .db 文件里
from langgraph.checkpoint.sqlite import SqliteSaver     # SqliteSaver 是 LangGraph 提供的一个 Checkpoint 保存器，负责把 Agent 每一步的状态写进 SQLite 数据库。



# 连接sqlite
connection = sqlite3.connect("resources/checkpoint.db", check_same_thread=False)
# 初始化checkpointer
checkpointer = SqliteSaver(connection)
# 自动建表
checkpointer.setup()        # 在数据库里创建 Checkpoint 所需的表结构（checkpoints、writes 等）。如果表已存在，不会重复创建，所以这行可以安全地每次运行。如果跳过这一步，第一次运行会因为找不到表而报错。

qwen_agent = create_agent(model=qwen_model, checkpointer=checkpointer, )


In [15]:
# 测试
from langchain.messages import HumanMessage

config={
    "configurable":{
        "thread_id":"user_2"
    }
}

# 第一次调用，告知AI我的信息
response = qwen_agent.invoke({"messages": [HumanMessage(content="你好，我叫YangRui，最喜欢柴犬")]}, config=config, )
print(response)

{'messages': [HumanMessage(content='你好，我叫YangRui，最喜欢柴犬', additional_kwargs={}, response_metadata={}, id='f9f853ce-2b93-4bf7-a4db-f243050c972e'), AIMessage(content='你好，YangRui！很高兴认识你。柴犬确实是非常可爱和迷人的狗狗，它们不仅外表讨人喜欢，性格也非常友好、忠诚。如果你有任何关于柴犬的问题，比如养护知识、训练技巧或者是寻找柴犬相关的信息，我都很乐意帮助你哦！另外，如果你愿意分享的话，我很想听听你是怎么开始喜欢上柴犬的？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 19, 'total_tokens': 98, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'qwen-max', 'system_fingerprint': None, 'id': 'chatcmpl-42964b6b-b565-983e-87ca-fad44ee93795', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a188a4-745d-7e80-88de-5f477aa8ce0d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 79, 'total_tokens': 98, 'input_token_details': {'cache_read

In [16]:
# 第二次调用，，询问我的信息，这次带上threa_id=user_2，看是否能够唤起记忆
response = qwen_agent.invoke({"messages": [HumanMessage(content="我最喜欢的动物是什么")]}, config=config, )
print(response)

{'messages': [HumanMessage(content='你好，我叫YangRui，最喜欢柴犬', additional_kwargs={}, response_metadata={}, id='f9f853ce-2b93-4bf7-a4db-f243050c972e'), AIMessage(content='你好，YangRui！很高兴认识你。柴犬确实是非常可爱和迷人的狗狗，它们不仅外表讨人喜欢，性格也非常友好、忠诚。如果你有任何关于柴犬的问题，比如养护知识、训练技巧或者是寻找柴犬相关的信息，我都很乐意帮助你哦！另外，如果你愿意分享的话，我很想听听你是怎么开始喜欢上柴犬的？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 19, 'total_tokens': 98, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'qwen-max', 'system_fingerprint': None, 'id': 'chatcmpl-42964b6b-b565-983e-87ca-fad44ee93795', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a188a4-745d-7e80-88de-5f477aa8ce0d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 79, 'total_tokens': 98, 'input_token_details': {'cache_read

# 4.记忆管理

当会话历史过长时，可能会超出模型的上下文窗口限制，常见的解决方案有：
- 修剪消息
- 删除消息
- 总结消息摘要

这里我们演示总结消息摘要的方案


In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

# 初始化checkpointer
checkpointer = InMemorySaver()

# 初始化中间件
# 没有在创建智能体时才去初始化中间件，那样的化代码都堆在一起不好看
middleware = SummarizationMiddleware(model=qwen_model, trigger=("messages", 3), keep=("messages", 1))
# 创建agent
agent = create_agent(model=qwen_model, checkpointer=checkpointer, middleware=[middleware], )

# 初始化RunnableConfig
config:RunnableConfig = {
    "configurable": {
        "thread_id": "user_3"
    }
}

# 制造长会话历史
agent.invoke({"messages": [HumanMessage(content="你好，我是虎哥.")]}, config)
agent.invoke({"messages": [HumanMessage(content="我最喜欢的运动是乒乓")]}, config)
agent.invoke({"messages": [HumanMessage(content="我最喜欢的动物是猫猫")]}, config)

# 测试消息
final_response = agent.invoke({"messages": [HumanMessage("你还记得我吗")]},config)



In [20]:
print(final_response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n用户虎哥尚未明确具体请求或目标，但表达了对乒乓球运动的喜爱。\n\n## SUMMARY\n- 用户最喜欢的运动是乒乓球。\n- 用户最喜欢的动物是猫猫。\n- AI询问了用户是否经常打乒乓球以及是否有特别想了解关于乒乓球的信息（如技术提高的方法、比赛规则、寻找球友和场地等）。\n- 用户未进一步说明具体需求或问题。\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\n等待虎哥提出具体的帮助请求，特别是与乒乓球相关的需求或问题。同时，如果用户对猫咪有兴趣了解更多，可以准备相关信息。', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='9c7d17c8-054b-4a38-b57e-ebef13b932d2'), HumanMessage(content='你还记得我吗', additional_kwargs={}, response_metadata={}, id='aa29ffb6-cb3f-4e67-93d4-d06315b761cf'), AIMessage(content='当然记得您，虎哥！我们之前聊过您对乒乓球的热爱以及您最喜欢的动物是猫咪。不知道接下来您是否有具体想了解的内容呢？无论是关于乒乓球的技术提高、比赛规则，还是关于猫咪的一些有趣知识或养护技巧，我都很乐意为您提供帮助。如果您有其他任何感兴趣的话题，也欢迎随时告诉我！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 148, 'total_tokens': 218, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': Non